# RAG sur le Code du travail français — Congés payés

Ce notebook démontre un pipeline **RAG (Retrieval-Augmented Generation)** complet et fonctionnel,
appliqué à un corpus réel : les articles du Code du travail français relatifs aux **congés payés**
(articles L3141-1 à L3141-28), récupérés depuis des sources juridiques publiques.

**Pipeline :**
1. **Corpus** : 14 articles réels du Code du travail (section congés payés).
2. **Retrieval** : vectorisation TF-IDF + similarité cosinus pour retrouver les articles pertinents
   à une question posée en langage naturel.
3. **Generation** : composition d'une réponse citée, strictement ancrée dans le texte légal récupéré
   (aucune hallucination possible : voir la note sur l'architecture dans le README).

Le module `rag_pipeline.py` est conçu pour être facilement étendu avec des embeddings denses
(sentence-transformers) et un modèle de génération neuronal (ex: flan-t5) — voir le README.

In [1]:
from pathlib import Path
from rag_pipeline import LegalRag

DATA_PATH = Path("data/corpus_conges_payes.json")
rag = LegalRag(DATA_PATH, embedding_backend="tfidf", generation_backend="extractive")

print(f"Corpus chargé : {len(rag.corpus)} articles")
print(f"Backend de récupération : {rag.retriever.backend}")
print(f"Backend de génération : {rag.generator.backend}")

Corpus chargé : 14 articles
Backend de récupération : tfidf
Backend de génération : extractive


## Aperçu du corpus

In [2]:
for a in rag.corpus:
    print(f"{a.titre:18s} | {len(a.texte):4d} caractères | {a.texte[:70]}...")

Article L3141-1    |   77 caractères | Tout salarié a droit chaque année à un congé payé à la charge de l'emp...
Article L3141-2    |  249 caractères | Les salariés de retour d'un congé de maternité prévu à l'article L. 12...
Article L3141-3    |  188 caractères | Le salarié a droit à un congé de deux jours et demi ouvrables par mois...
Article L3141-4    |  166 caractères | Sont assimilées à un mois de travail effectif pour la détermination de...
Article L3141-5    | 1165 caractères | Sont considérées comme périodes de travail effectif pour la déterminat...
Article L3141-6    |  147 caractères | L'absence du salarié ne peut avoir pour effet d'entraîner une réductio...
Article L3141-8    |  757 caractères | Les salariés de moins de vingt et un ans au 30 avril de l'année précéd...
Article L3141-9    |   94 caractères | La durée du congé annuel peut être majorée en raison de l'âge, de l'an...
Article L3141-12   |  246 caractères | Les congés peuvent être pris dès l'embauche, sans préjudi

## Exemples de questions / réponses

In [3]:
questions = [
    "Combien de jours de congés payés acquiert-on par mois de travail ?",
    "Un salarié peut-il prendre ses congés dès son embauche ?",
    "Que se passe-t-il pour les congés payés si le contrat de travail est rompu ?",
    "Comment est fixé l'ordre des départs en congés ?",
    "Un salarié a-t-il droit à des jours de congés supplémentaires pour ses enfants ?",
]

for q in questions:
    result = rag.ask(q, k=3)
    print("=" * 100)
    print("Q:", result.question)
    print("-" * 100)
    print("Réponse:", result.reponse)
    print()
    print("Articles récupérés (score de similarité) :")
    for r in result.articles_cites:
        print(f"  - {r.article.titre} (score={r.score:.3f})")
    print()

Q: Combien de jours de congés payés acquiert-on par mois de travail ?
----------------------------------------------------------------------------------------------------
Réponse: D'après l'Article L3141-4 du Code du travail : « Sont assimilées à un mois de travail effectif pour la détermination de la durée du congé les périodes équivalentes à quatre semaines ou vingt-quatre jours de travail. » (voir aussi : L3141-3, L3141-24).

Articles récupérés (score de similarité) :
  - Article L3141-4 (score=0.232)
  - Article L3141-3 (score=0.226)
  - Article L3141-24 (score=0.110)

Q: Un salarié peut-il prendre ses congés dès son embauche ?
----------------------------------------------------------------------------------------------------
Réponse: D'après l'Article L3141-12 du Code du travail : « Les congés peuvent être pris dès l'embauche, sans préjudice des règles de détermination de la période de prise des congés et de l'ordre des départs et des règles de fractionnement du congé fixées dans

## Inspection du retrieval : quels articles sont jugés similaires ?

On peut inspecter la matrice de similarité pour comprendre pourquoi un article est retrouvé
(ou pas) pour une question donnée — utile pour diagnostiquer les limites du TF-IDF (approche
par mots-clés, qui ne capture pas toujours le lien sémantique entre une question et un article
qui ne partage pas de vocabulaire explicite).

In [4]:
import numpy as np

question_test = "Les congés payés comptent-ils comme du temps de travail effectif ?"
retrieved = rag.retriever.retrieve(question_test, k=len(rag.corpus))

print("Question :", question_test)
print()
print("Classement de TOUS les articles par similarité :")
for r in retrieved:
    print(f"  {r.score:.3f}  {r.article.titre}")

Question : Les congés payés comptent-ils comme du temps de travail effectif ?

Classement de TOUS les articles par similarité :
  0.168  Article L3141-4
  0.162  Article L3141-24
  0.113  Article L3141-12
  0.106  Article L3141-5
  0.104  Article L3141-3
  0.054  Article L3141-15
  0.050  Article L3141-13
  0.040  Article L3141-28
  0.032  Article L3141-9
  0.022  Article L3141-8
  0.020  Article L3141-2
  0.019  Article L3141-6
  0.017  Article L3141-19
  0.010  Article L3141-1


### Limite observée

Le TF-IDF est une méthode par mots-clés : il peut manquer des articles sémantiquement liés
mais formulés différemment (par exemple l'article L3141-5, qui liste les périodes assimilées
à du travail effectif, peut être mal classé si la question n'emploie pas un vocabulaire proche).
C'est une limite connue et documentée de l'approche "bag-of-words" — la piste d'amélioration
naturelle est de passer à des embeddings denses (sentence-transformers), ce que le code supporte
déjà via `embedding_backend="sentence-transformers"` (voir le README pour l'activer).

## Avertissement sur le périmètre

Ce projet est un démonstrateur pédagogique sur un périmètre volontairement restreint
(congés payés) afin de garder un corpus manipulable et des réponses vérifiables. Il ne constitue
pas un conseil juridique et ne couvre pas l'intégralité du Code du travail. Toutes les réponses
du mode "extractif" sont des citations directes du texte légal récupéré — elles ne peuvent donc
pas contenir d'information inventée.